# Liu2024 MOABB / S-JEPA-Style CSP/FBCSP Baseline

This notebook keeps the preprocessing/windowing close to your S-JEPA downstream setup, but evaluates classical CSP/FBCSP baselines.

Purpose:
- Load Liu2024 through MOABB.
- Use S-JEPA-style preprocessing: pick EEG, resample to 128 Hz, bandpass 0.5–40 Hz, average reference.
- Create explicit fixed-length windows, default **537 samples** = 4.1953125s at 128 Hz.
- Run within-subject 5-fold CSP+LDA and FBCSP+LDA.
- Print overlap diagnostics so you can see whether the classical result is affected by Liu's dense annotations.

This is **not** expected to exactly match the Liu paper's 55.57% CSP+LDA baseline, because the paper used the original epoched data and a different validation protocol.


# 1. Setup

In [1]:
import os
import sys
import json
import platform
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.signal import butter, sosfiltfilt

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

try:
    import mne
    from mne.decoding import CSP
except Exception as exc:
    raise RuntimeError('This notebook needs MNE installed: pip install mne') from exc

try:
    from braindecode.datasets import MOABBDataset, BaseConcatDataset
    from braindecode.preprocessing import Preprocessor, preprocess, create_windows_from_events
except Exception as exc:
    raise RuntimeError('This notebook needs Braindecode and MOABB installed.') from exc

print('Runtime Environment:')
print(f'  Python:   {sys.version}')
print(f'  Platform: {platform.platform()}')
print(f'  MNE:      {mne.__version__}')
print(f'  Workdir:  {Path.cwd()}')


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Runtime Environment:
  Python:   3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
  Platform: macOS-26.2-arm64-arm-64bit
  MNE:      1.11.0
  Workdir:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src


# 2. Constants

In [2]:
WORKING_DIR = Path.cwd()
ARTIFACT_DIR = WORKING_DIR / 'artifacts' / 'liu2024-sjepa-style-csp-fbcsp' / datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'

DATASET_NAME = 'Liu2024'
SUBJECTS = list(range(1, 51))
LABELS_TO_KEEP = ['left_hand', 'right_hand']
MAPPING = {'left_hand': 0, 'right_hand': 1}

SFREQ = 128
BANDPASS_LOW = 0.5
BANDPASS_HIGH = 40.0
TARGET_WINDOW_SAMPLES = 537
TARGET_WINDOW_DURATION_S = TARGET_WINDOW_SAMPLES / SFREQ
CV_FOLDS = 5
RANDOM_STATE = 2026

# For S-JEPA comparison we keep all events and allow Braindecode to warn about overlap.
# This mirrors the fact that S-JEPA training/classification is usually run on fixed windows,
# but it should be interpreted carefully for CSP.
ON_OVERLAPPING_EVENTS = 'warn'

CSP_BAND = (8.0, 30.0)
FBCSP_BANDS = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 32), (32, 36), (36, 40)]
N_CSP_COMPONENTS = 4

print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Target window: {TARGET_WINDOW_SAMPLES} samples = {TARGET_WINDOW_DURATION_S:.6f}s')


Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-sjepa-style-csp-fbcsp/20260518_152240
Target window: 537 samples = 4.195312s


# 3. Logger

In [3]:
_log_handle = open(LOG_PATH, 'w', buffering=1)

def log(msg=''):
    text = str(msg)
    print(text)
    _log_handle.write(text + '\n')

log('Liu2024 S-JEPA-style CSP/FBCSP run')
log(f'Artifacts: {ARTIFACT_DIR}')
log(f'Target window: {TARGET_WINDOW_SAMPLES} samples = {TARGET_WINDOW_DURATION_S:.6f}s')


Liu2024 S-JEPA-style CSP/FBCSP run
Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-sjepa-style-csp-fbcsp/20260518_152240
Target window: 537 samples = 4.195312s


# 4. Load Liu2024 through MOABB

In [4]:
dataset = MOABBDataset(dataset_name=DATASET_NAME, subject_ids=SUBJECTS)
log(f'Loaded MOABBDataset {DATASET_NAME}')
log(f'Recordings: {len(dataset.datasets)}')

# Basic raw summary before preprocessing.
rows = []
for i, ds in enumerate(dataset.datasets):
    raw = ds.raw
    desc = list(raw.annotations.description)
    vals, counts = np.unique(desc, return_counts=True) if len(desc) else ([], [])
    rows.append({
        'recording_index': i,
        'subject': ds.description.get('subject', None),
        'session': ds.description.get('session', None),
        'run': ds.description.get('run', None),
        'sfreq': raw.info['sfreq'],
        'n_channels': len(raw.ch_names),
        'n_times': raw.n_times,
        'duration_s': raw.n_times / raw.info['sfreq'],
        'annotation_counts': dict(zip([str(v) for v in vals], [int(c) for c in counts])),
    })
raw_summary = pd.DataFrame(rows)
display(raw_summary.head())
raw_summary.to_csv(ARTIFACT_DIR / 'raw_recording_summary.csv', index=False)
log(raw_summary.head(20).to_string(index=False))


1.4.3
Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


...

/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]
Loaded MOABBDataset Liu2024
Recordings: 50


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


,recording_index,subject,session,run,sfreq,n_channels,n_times,duration_s,annotation_counts
0,0,1,0,0,500.0,32,160000,320.0,"{'left_hand': 20, 'right_hand': 19}"
1,1,2,0,0,500.0,32,160000,320.0,"{'left_hand': 20, 'right_hand': 19}"
2,2,3,0,0,500.0,32,160000,320.0,"{'left_hand': 20, 'right_hand': 19}"
3,3,4,0,0,500.0,32,160000,320.0,"{'left_hand': 20, 'right_hand': 19}"
4,4,5,0,0,500.0,32,160000,320.0,"{'left_hand': 20, 'right_hand': 19}"


 recording_index  subject session run  sfreq  n_channels  n_times  duration_s                   annotation_counts
               0        1       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               1        2       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               2        3       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               3        4       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               4        5       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               5        6       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               6        7       0   0  500.0          32   160000       320.0 {'left_hand': 20, 'right_hand': 19}
               7        8       0   0  500.0          32   160000       320.0 {'left_han

# 5. S-JEPA-style preprocessing

In [5]:
preprocessors = [
    Preprocessor('pick_types', eeg=True, eog=False, stim=False, misc=False, verbose=False),
    Preprocessor('resample', sfreq=SFREQ, verbose=False),
    Preprocessor('filter', l_freq=BANDPASS_LOW, h_freq=BANDPASS_HIGH, verbose=False),
    Preprocessor(lambda raw: raw.set_eeg_reference('average', projection=False, verbose=False), apply_on_array=False),
]

log('Applying preprocessing: pick EEG -> resample 128 Hz -> filter 0.5-40 Hz -> average reference')
preprocess(dataset, preprocessors, n_jobs=1)
log('Preprocessing complete.')


Applying preprocessing: pick EEG -> resample 128 Hz -> filter 0.5-40 Hz -> average reference


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:77: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:75: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Preprocessing complete.


# 6. Annotation and overlap diagnostics after preprocessing

In [6]:
def summarize_annotations(ds_concat, labels_to_keep=LABELS_TO_KEEP):
    rows = []
    for rec_i, ds in enumerate(ds_concat.datasets):
        raw = ds.raw
        sfreq = float(raw.info['sfreq'])
        ann = raw.annotations
        for onset, duration, desc in zip(ann.onset, ann.duration, ann.description):
            if str(desc) in labels_to_keep:
                rows.append({
                    'recording_index': rec_i,
                    'subject': ds.description.get('subject', None),
                    'session': ds.description.get('session', None),
                    'run': ds.description.get('run', None),
                    'description': str(desc),
                    'onset_s': float(onset),
                    'duration_s': float(duration),
                    'duration_samples_round': int(round(float(duration) * sfreq)),
                    'duration_samples_floor': int(np.floor(float(duration) * sfreq)),
                    'sfreq': sfreq,
                })
    df = pd.DataFrame(rows)
    return df

ann_df = summarize_annotations(dataset)
ann_df.to_csv(ARTIFACT_DIR / 'selected_annotations_after_preprocessing.csv', index=False)
display(ann_df.head())

log('Annotation duration summary after preprocessing:')
log(ann_df.groupby('description')['duration_s'].describe().to_string())

# Consecutive event gap/overlap diagnostics within each recording.
overlap_rows = []
for (rec_i), g in ann_df.sort_values(['recording_index', 'onset_s']).groupby('recording_index'):
    g = g.reset_index(drop=True)
    for idx in range(len(g) - 1):
        cur = g.iloc[idx]
        nxt = g.iloc[idx + 1]
        gap = float(nxt['onset_s'] - cur['onset_s'])
        overlap = max(0.0, TARGET_WINDOW_DURATION_S - gap)
        overlap_rows.append({
            'recording_index': int(rec_i),
            'subject': cur['subject'],
            'current_label': cur['description'],
            'next_label': nxt['description'],
            'current_onset_s': cur['onset_s'],
            'next_onset_s': nxt['onset_s'],
            'onset_gap_s': gap,
            'window_duration_s': TARGET_WINDOW_DURATION_S,
            'overlap_s': overlap,
            'overlap_fraction': overlap / TARGET_WINDOW_DURATION_S,
        })
overlap_df = pd.DataFrame(overlap_rows)
overlap_df.to_csv(ARTIFACT_DIR / 'window_overlap_diagnostics.csv', index=False)

summary = {
    'n_pairs': int(len(overlap_df)),
    'n_any_overlap': int((overlap_df['overlap_s'] > 0).sum()),
    'fraction_any_overlap': float((overlap_df['overlap_s'] > 0).mean()),
    'n_major_overlap_gt_25pct': int((overlap_df['overlap_fraction'] > 0.25).sum()),
    'fraction_major_overlap_gt_25pct': float((overlap_df['overlap_fraction'] > 0.25).mean()),
    'min_gap_s': float(overlap_df['onset_gap_s'].min()),
    'median_gap_s': float(overlap_df['onset_gap_s'].median()),
    'max_gap_s': float(overlap_df['onset_gap_s'].max()),
}
log('Overlap summary for target S-JEPA-style window:')
log(json.dumps(summary, indent=2))
pd.DataFrame([summary]).to_csv(ARTIFACT_DIR / 'window_overlap_summary.csv', index=False)
display(pd.DataFrame([summary]))
display(overlap_df.sort_values('overlap_fraction', ascending=False).head(20))


,recording_index,subject,session,run,description,onset_s,duration_s,duration_samples_round,duration_samples_floor,sfreq
0,0,1,0,0,left_hand,4.006,4.0,512,512,128.0
1,0,1,0,0,right_hand,8.010,4.0,512,512,128.0
2,0,1,0,0,left_hand,10.000,4.0,512,512,128.0
3,0,1,0,0,right_hand,12.004,4.0,512,512,128.0
4,0,1,0,0,left_hand,16.006,4.0,512,512,128.0


Annotation duration summary after preprocessing:
              count  mean  std  min  25%  50%  75%  max
description                                            
left_hand    1000.0   4.0  0.0  4.0  4.0  4.0  4.0  4.0
right_hand    950.0   4.0  0.0  4.0  4.0  4.0  4.0  4.0
Overlap summary for target S-JEPA-style window:
{
  "n_pairs": 1900,
  "n_any_overlap": 1900,
  "fraction_any_overlap": 1.0,
  "n_major_overlap_gt_25pct": 1251,
  "fraction_major_overlap_gt_25pct": 0.6584210526315789,
  "min_gap_s": 0.001999999999995339,
  "median_gap_s": 2.0060000000000002,
  "max_gap_s": 4.004000000000005
}


,n_pairs,n_any_overlap,fraction_any_overlap,n_major_overlap_gt_25pct,fraction_major_overlap_gt_25pct,min_gap_s,median_gap_s,max_gap_s
0,1900,1900,1.0,1251,0.658421,0.002,2.006,4.004


,recording_index,subject,current_label,next_label,current_onset_s,next_onset_s,onset_gap_s,window_duration_s,overlap_s,overlap_fraction
1088,28,29,left_hand,right_hand,66.000,66.002,0.002,4.195312,4.193313,0.999523
1078,28,29,left_hand,right_hand,42.000,42.002,0.002,4.195312,4.193312,0.999523
735,19,20,right_hand,left_hand,40.004,40.044,0.040,4.195312,4.155313,0.990466
203,5,6,right_hand,left_hand,40.936,42.000,1.064,4.195312,3.131312,0.746384
212,5,6,left_hand,right_hand,64.914,66.000,1.086,4.195312,3.109313,0.741140
224,5,6,left_hand,right_hand,96.902,98.000,1.098,4.195312,3.097313,0.738279
197,5,6,right_hand,left_hand,24.896,26.000,1.104,4.195312,3.091313,0.736849
215,5,6,right_hand,left_hand,72.896,74.000,1.104,4.195312,3.091313,0.736849
227,5,6,right_hand,left_hand,104.884,106.000,1.116,4.195312,3.079313,0.733989
221,5,6,right_hand,left_hand,88.852,90.000,1.148,4.195312,3.047313,0.726361


# 7. Filter annotations to motor labels

In [7]:
def filter_annotations_by_description(ds_concat, descriptions_to_keep):
    descriptions_to_keep = set(descriptions_to_keep)
    total_before = 0
    total_after = 0
    for ds in ds_concat.datasets:
        raw = ds.raw
        ann = raw.annotations
        total_before += len(ann)
        keep = [i for i, desc in enumerate(ann.description) if str(desc) in descriptions_to_keep]
        raw.set_annotations(ann[keep])
        total_after += len(raw.annotations)
    return total_before, total_after

before, after = filter_annotations_by_description(dataset, LABELS_TO_KEEP)
log(f'Filtered annotations: kept {after} / {before}')


Filtered annotations: kept 1950 / 1950


# 8. Create explicit 537-sample windows

In [8]:
def compute_stop_offset_samples(ds_concat, target_window_samples=TARGET_WINDOW_SAMPLES):
    rows = summarize_annotations(ds_concat)
    if rows.empty:
        raise RuntimeError('No selected annotations found.')
    duration_samples = rows['duration_samples_floor'].to_numpy(dtype=int)
    min_duration_samples = int(duration_samples.min())
    median_duration_samples = int(np.median(duration_samples))
    stop_offset = int(target_window_samples - min_duration_samples)
    log('Window offset calculation:')
    log(f'  min annotation samples floor: {min_duration_samples}')
    log(f'  median annotation samples:    {median_duration_samples}')
    log(f'  target_window_samples:       {target_window_samples}')
    log(f'  trial_stop_offset_samples:   {stop_offset}')
    return stop_offset

trial_stop_offset_samples = compute_stop_offset_samples(dataset)

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=trial_stop_offset_samples,
    mapping=MAPPING,
    window_size_samples=TARGET_WINDOW_SAMPLES,
    window_stride_samples=TARGET_WINDOW_SAMPLES,
    drop_last_window=True,
    preload=True,
    on_overlapping_events=ON_OVERLAPPING_EVENTS,
)

log(f'Created windows dataset with {len(windows_dataset)} windows.')


Window offset calculation:
  min annotation samples floor: 512
  median annotation samples:    512
  target_window_samples:       537
  trial_stop_offset_samples:   25
Created windows dataset with 1950 windows.


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/braindecode/preprocessing/windowers.py:769: UserWarning: Overlapping trials detected. You can ignore, warn, or raise an error, using the on_overlapping_events argument.
  warnings.warn(msg)


# 9. Extract X/y/subject arrays

In [9]:
Xs, ys, subjects_arr = [], [], []
for ds in windows_dataset.datasets:
    X = ds.windows.get_data()  # windows x channels x samples
    y = np.asarray(ds.y, dtype=int)
    subj = ds.description.get('subject', None)
    Xs.append(X)
    ys.append(y)
    subjects_arr.extend([subj] * len(y))

X_all = np.concatenate(Xs, axis=0)
y_all = np.concatenate(ys, axis=0)
subjects_arr = np.asarray(subjects_arr)

log(f'X_all shape: {X_all.shape}')
log(f'y_all counts: {np.bincount(y_all, minlength=2).tolist()}')
window_counts = pd.DataFrame({'subject': subjects_arr, 'y': y_all}).groupby('subject')['y'].agg(['count', lambda s: np.bincount(s, minlength=2).tolist()])
window_counts.columns = ['n_windows', 'class_counts']
display(window_counts.head())
window_counts.to_csv(ARTIFACT_DIR / 'window_counts_by_subject.csv')


AttributeError: 'EEGWindowsDataset' object has no attribute 'windows'

# 10. Classical model helpers

In [ ]:
def bandpass_zero_phase(X, sfreq, l_freq, h_freq, order=5):
    sos = butter(order, [l_freq, h_freq], btype='bandpass', fs=sfreq, output='sos')
    return sosfiltfilt(sos, X, axis=-1)

def make_csp_lda(n_components=N_CSP_COMPONENTS):
    return Pipeline([
        ('csp', CSP(n_components=n_components, reg='ledoit_wolf', log=True, norm_trace=False)),
        ('lda', LinearDiscriminantAnalysis()),
    ])

def make_fbcsp_features(X_train, y_train, X_test):
    feats_train, feats_test = [], []
    for band in FBCSP_BANDS:
        Xtr = bandpass_zero_phase(X_train, SFREQ, band[0], band[1])
        Xte = bandpass_zero_phase(X_test, SFREQ, band[0], band[1])
        csp = CSP(n_components=N_CSP_COMPONENTS, reg='ledoit_wolf', log=True, norm_trace=False)
        feats_train.append(csp.fit_transform(Xtr, y_train))
        feats_test.append(csp.transform(Xte))
    return np.concatenate(feats_train, axis=1), np.concatenate(feats_test, axis=1)

def fold_metrics(y_test, pred, scores=None):
    row = {
        'accuracy': float(accuracy_score(y_test, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
        'confusion_matrix': confusion_matrix(y_test, pred, labels=[0, 1]).tolist(),
        'prediction_histogram': np.bincount(pred, minlength=2).tolist(),
    }
    if scores is not None and len(np.unique(y_test)) == 2:
        try:
            row['roc_auc'] = float(roc_auc_score(y_test, scores))
        except Exception:
            row['roc_auc'] = None
    else:
        row['roc_auc'] = None
    return row


# 11. Within-subject 5-fold evaluation

In [ ]:
rows = []
for subj in sorted(pd.unique(subjects_arr), key=lambda x: int(x)):
    idx = np.where(subjects_arr == subj)[0]
    X = X_all[idx]
    y = y_all[idx]
    counts = np.bincount(y, minlength=2)
    log(f'Subject {subj}: X={X.shape}, class_counts={counts.tolist()}')
    if counts.min() < CV_FOLDS:
        log(f'  SKIP subject {subj}: not enough trials per class for {CV_FOLDS}-fold CV')
        continue
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
        X_train_raw, X_test_raw = X[tr], X[te]
        y_train, y_test = y[tr], y[te]

        # CSP band for CSP+LDA.
        X_train = bandpass_zero_phase(X_train_raw, SFREQ, CSP_BAND[0], CSP_BAND[1])
        X_test = bandpass_zero_phase(X_test_raw, SFREQ, CSP_BAND[0], CSP_BAND[1])
        clf = make_csp_lda()
        clf.fit(X_train, y_train)
        pred = clf.predict(X_test)
        scores = clf.decision_function(X_test) if hasattr(clf, 'decision_function') else None
        row = {
            'subject_id': str(subj),
            'fold_id': fold,
            'model_name': 'CSP_LDA',
            'n_train': int(len(tr)),
            'n_test': int(len(te)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
        }
        row.update(fold_metrics(y_test, pred, scores))
        rows.append(row)

        # FBCSP + LDA to match your prior baseline family.
        Ftr, Fte = make_fbcsp_features(X_train_raw, y_train, X_test_raw)
        lda = LinearDiscriminantAnalysis()
        lda.fit(Ftr, y_train)
        pred = lda.predict(Fte)
        scores = lda.decision_function(Fte) if hasattr(lda, 'decision_function') else None
        row = {
            'subject_id': str(subj),
            'fold_id': fold,
            'model_name': 'FBCSP_LDA',
            'n_train': int(len(tr)),
            'n_test': int(len(te)),
            'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
            'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
            'n_filter_bands': len(FBCSP_BANDS),
        }
        row.update(fold_metrics(y_test, pred, scores))
        rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv(ARTIFACT_DIR / 'cv_results.csv', index=False)
display(results_df.head())
log(f'Wrote fold results: {ARTIFACT_DIR / "cv_results.csv"}')


# 12. Summaries and collapse diagnostics

In [ ]:
global_metrics = (
    results_df
    .groupby('model_name')
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        std_balanced_accuracy=('balanced_accuracy', 'std'),
        mean_roc_auc=('roc_auc', 'mean'),
        std_roc_auc=('roc_auc', 'std'),
        n_folds=('accuracy', 'size'),
    )
    .reset_index()
)

collapse_rows = []
for model, g in results_df.groupby('model_name'):
    collapsed = 0
    pred0 = pred1 = 0
    for hist in g['prediction_histogram']:
        if isinstance(hist, str):
            hist = json.loads(hist)
        if hist[0] == 0 or hist[1] == 0:
            collapsed += 1
            if hist[1] == 0:
                pred0 += 1
            if hist[0] == 0:
                pred1 += 1
    collapse_rows.append({
        'model_name': model,
        'n_folds': int(len(g)),
        'n_collapsed_single_class': int(collapsed),
        'collapsed_fraction': float(collapsed / len(g)),
        'collapsed_to_class_0': int(pred0),
        'collapsed_to_class_1': int(pred1),
    })
collapse_df = pd.DataFrame(collapse_rows)

display(global_metrics)
display(collapse_df)

global_metrics.to_csv(ARTIFACT_DIR / 'global_metrics.csv', index=False)
collapse_df.to_csv(ARTIFACT_DIR / 'collapse_diagnostics.csv', index=False)

metadata = {
    'notebook': 'liu2024_sjepa_style_moabb_csp_fbcsp',
    'dataset_source': 'MOABB',
    'sfreq': SFREQ,
    'bandpass_preprocess': [BANDPASS_LOW, BANDPASS_HIGH],
    'target_window_samples': TARGET_WINDOW_SAMPLES,
    'target_window_duration_s': TARGET_WINDOW_DURATION_S,
    'on_overlapping_events': ON_OVERLAPPING_EVENTS,
    'trial_stop_offset_samples': int(trial_stop_offset_samples),
    'cv_folds': CV_FOLDS,
    'csp_band': CSP_BAND,
    'fbcsp_bands': FBCSP_BANDS,
    'overlap_summary': summary,
}
with open(ARTIFACT_DIR / 'run_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

log('\nGlobal metrics:')
log(global_metrics.to_string(index=False))
log('\nCollapse diagnostics:')
log(collapse_df.to_string(index=False))
log(f'\nArtifacts saved to: {ARTIFACT_DIR}')
_log_handle.close()


# 13. Interpretation note

Use this notebook to compare directly with your S-JEPA setup. If the paper-mimic notebook gives a more reasonable CSP result but this notebook struggles, the culprit is probably not the classifier implementation alone. It is the combination of MOABB event annotations, resampling, 537-sample windows, dense overlap, and within-subject 5-fold validation.
